# Notebook 05 — Transcription and Translation Mechanics

**Module:** 07 — Biochemistry and Molecular Biology  
**Notebook:** 05 of 10  
**Prerequisites:** Module 05 NB01–NB03 (central dogma overview)  
**Time estimate:** 75 minutes

> **RNA-seq connection:** Junction-spanning reads, intron retention artefacts, and
> 3'UTR-biased coverage all trace directly to the splicing and polyadenylation
> mechanics covered here.

---
## Step 1 — Motivation

Module 05 covered the central dogma as an overview. This notebook goes one level
deeper: the specific molecular mechanisms of transcription and translation that
produce the data shapes we see in RNA-seq, ribosome profiling, and proteomics
experiments. Understanding *why* a spliceosome exists is what lets you understand
*why* STAR (the RNA-seq aligner) needs to detect exon-exon junctions.

---
## Step 3 — Biological Background

### Transcription

**Promoter elements (eukaryotes):**
- **TATA box** (−30): TATAAA sequence; binds TBP (TATA-binding protein)
- **Initiator element** (Inr): at the transcription start site (TSS)
- **GC box** (−90): GGGCGG; binds Sp1
- **Enhancers:** distal elements (up to 1 Mb away) that loop to the promoter

**RNA Polymerase II (Pol II):**
- Synthesises pre-mRNA
- CTD (C-terminal domain): repeated heptapeptide YSPTSPS; phosphorylation pattern
  encodes transcription stage (Ser5-P = initiation; Ser2-P = elongation)

**Splicing (GT-AG rule):**
- Introns begin GU (5' splice site) and end AG (3' splice site)
- Spliceosome: 5 snRNAs (U1, U2, U4, U5, U6) + ~150 proteins
- Branch point A (usually ~30 nt upstream of 3' SS) attacks 5' SS → lariat intermediate
- Second transesterification: exon ligation + lariat excision

**Alternative splicing:**
~95% of multi-exon human genes undergo alternative splicing. Modes:
exon skipping (most common), intron retention, alternative 5'/3' splice sites,
mutually exclusive exons.

### Translation

**Ribosome:** 80S (eukaryotic) = 40S + 60S subunits
- **A site:** aminoacyl tRNA docking
- **P site:** peptidyl tRNA (growing chain attached)
- **E site:** exit site for empty tRNA

**Elongation cycle:** (1) aminoacyl-tRNA selection at A site → (2) peptide bond
formation (peptidyl transferase, a ribozyme) → (3) translocation (A→P, P→E)

**Codon usage bias:**
Synonymous codons are not used equally. Organisms optimise codon usage to match
tRNA availability — highly expressed genes use 'optimal' codons. Codon Adaptation
Index (CAI) = geometric mean of relative synonymous codon usage (RSCU) for each
codon in the sequence.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
# Cell 6.1 — Splice site detection (GT-AG rule)
def find_splice_sites(pre_mrna: str) -> list:
    """
    Find candidate 5' (GT) and 3' (AG) splice sites in a pre-mRNA sequence.
    Returns list of (position, type) tuples.
    """
    seq = pre_mrna.upper().replace('U', 'T')  # work in DNA space
    sites = []
    for i in range(len(seq) - 1):
        dimer = seq[i:i+2]
        if dimer == 'GT':
            sites.append((i, "5'SS (GT)"))
        elif dimer == 'AG':
            sites.append((i, "3'SS (AG)"))
    return sites

# Simulated pre-mRNA: exon1|intron|exon2 (GT starts intron, AG ends it)
pre_mrna = 'ATGCGATCGATCG' + 'GTAAGCTAGCTAGCTAGCAG' + 'CGATCGATCGATCGATG'
#                              ^^intron start (GT)    ^^intron end (AG)
sites = find_splice_sites(pre_mrna)
print(f"Candidate splice sites in example pre-mRNA:")
for pos, site_type in sites:
    context = pre_mrna[max(0,pos-3):pos+5]
    print(f"  pos {pos:>3}: {site_type}  context: ...{context}...")

print("\nTrue intron: positions 13–32 (GT at 13, AG at 31)")

In [ ]:
# Cell 6.2 — Codon usage analysis and CAI
GENETIC_CODE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGT':'S','AGC':'S','AGA':'R','AGG':'R','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}

def codon_usage(cds: str) -> dict:
    """Compute codon frequencies from a coding sequence (multiple of 3)."""
    cds = cds.upper()
    codons = [cds[i:i+3] for i in range(0, len(cds)-2, 3)
              if cds[i:i+3] in GENETIC_CODE and GENETIC_CODE[cds[i:i+3]] != '*']
    return Counter(codons)

def rscu(usage: dict) -> dict:
    """
    Relative Synonymous Codon Usage (RSCU).
    RSCU = observed / expected (if all synonymous codons used equally).
    """
    from itertools import groupby

    # Group codons by amino acid
    aa_codons = {}
    for codon, aa in GENETIC_CODE.items():
        if aa == '*':
            continue
        aa_codons.setdefault(aa, []).append(codon)

    rscu_vals = {}
    for aa, codons in aa_codons.items():
        n_syn = len(codons)  # number of synonymous codons for this AA
        total = sum(usage.get(c, 0) for c in codons)
        for codon in codons:
            obs = usage.get(codon, 0)
            expected = total / n_syn
            rscu_vals[codon] = obs / expected if expected > 0 else 0.0
    return rscu_vals

# Example: highly expressed E. coli gene (GFP optimised codon)
gfp_opt = 'ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCCTGGTCGAGCTGGACGGCGACGTAAACGGCCACAAGTTCAGCGTGTCCGGCGAGGGCGAGGGCGATGCCACCTACGGCAAGCTGACCCTGAAGTTCATCTGCACCACCGGCAAGCTGCCCGTGCCCTGGCCCACCCTCGTGACCACCCTGACCTACGGCGTGCAGTGCTTCAGCCGCTACCCCGACCACATGAAGCAGCACGACTTCTTCAAGTCCGCCATGCCCGAAGGCTACGTCCAGGAGCGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCGCCGAGGTGAAGTTCGAGGGCGACACCCTGGTGAACCGCATCGAGCTGAAGGGCATCGACTTCAAGGAGGACGGCAACATCCTGGGGCACAAGCTGGAGTACAACTACAACAGCCACAACGTCTATATCATGGCCGACAAGCAGAAGAACGGCATCAAGGTGAACTTCAAGATCCGCCACAACATCGAGGACGGCAGCGTGCAGCTCGCCGACCACTACCAGCAGAACACCCCCATCGGCGACGGCCCCGTGCTGCTGCCCGACAACCACTACCTGAGCACCCAGTCCGCCCTGAGCAAAGACCCCAACGAGAAGCGCGATCACATGGTCCTGCTGGAGTTCGTGACCGCCGCCGGGATCACTCTCGGCATGGACGAGCTGTACAAGTAA'
usage = codon_usage(gfp_opt)
rscu_vals = rscu(usage)

print(f"GFP codon usage (first 5 most-used codons):")
top_codons = sorted(usage.keys(), key=lambda c: -usage[c])[:5]
for c in top_codons:
    aa = GENETIC_CODE[c]
    print(f"  {c} ({aa}): used {usage[c]} times, RSCU = {rscu_vals[c]:.2f}")

In [ ]:
# Cell 6.3 — Simulate ribosome profiling footprint
def simulate_ribosome_profiling(cds: str, n_ribosomes: int = 200,
                                  footprint_len: int = 30, seed: int = 42) -> np.ndarray:
    """
    Simulate ribosome profiling (Ribo-seq) coverage.
    Ribosomes protect ~30 nt footprints; denser at slow codons.
    Returns per-nucleotide coverage array.
    """
    rng = np.random.default_rng(seed)
    L = len(cds)
    coverage = np.zeros(L)

    # Positions weighted by codon rarity (rare codons = slower ribosome)
    n_codons = L // 3
    weights = np.ones(n_codons)

    # Make rare codons (low RSCU) create ribosome pauses (higher weight)
    usage_cds = codon_usage(cds)
    rscu_cds = rscu(usage_cds)
    for i in range(n_codons):
        codon = cds[3*i:3*i+3].upper()
        r = rscu_cds.get(codon, 1.0)
        weights[i] = max(0.1, 2.0 - r)  # rare codons (low RSCU) → higher weight
    weights /= weights.sum()

    # Place ribosomes
    positions = rng.choice(n_codons, size=n_ribosomes, p=weights)
    for pos in positions:
        nt_start = pos * 3
        nt_end = min(L, nt_start + footprint_len)
        coverage[nt_start:nt_end] += 1

    return coverage

coverage = simulate_ribosome_profiling(gfp_opt, n_ribosomes=500)
print(f"Ribosome profiling simulation:")
print(f"  Total covered nucleotides: {(coverage > 0).sum()}/{len(gfp_opt)}")
print(f"  Peak coverage position: {np.argmax(coverage)} nt")
print(f"  Average coverage: {coverage.mean():.2f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Codon usage bias + Ribosome profiling coverage
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel 1: RSCU values for a subset of codons
ax = axes[0]
# Show RSCU for Leucine codons (6 synonymous) — most degenerate AA
leu_codons = [c for c, aa in GENETIC_CODE.items() if aa == 'L']
leu_rscu = [rscu_vals.get(c, 0) for c in leu_codons]

colors_rscu = ['tomato' if r > 1.5 else ('steelblue' if r > 0.5 else 'lightgray')
               for r in leu_rscu]
bars = ax.bar(leu_codons, leu_rscu, color=colors_rscu, alpha=0.8)
ax.axhline(1.0, color='black', ls='--', lw=1, label='Expected (equal usage)')
ax.set_ylabel('RSCU (>1 = over-represented)')
ax.set_title('Codon usage bias: Leucine (6 synonymous codons)\nGFP codon-optimised sequence')
ax.legend(fontsize=8)

# Panel 2: Ribosome profiling coverage with 3-nt periodicity
ax = axes[1]
x = np.arange(len(coverage))
ax.fill_between(x[:300], coverage[:300], color='steelblue', alpha=0.5)
ax.plot(x[:300], coverage[:300], color='steelblue', lw=0.8)

# Highlight 3-nt periodicity (every 3rd position = P-site)
p_sites = np.zeros(300)
for i in range(0, 300, 3):
    p_sites[i] = coverage[i]
ax.scatter(x[:300][::3], coverage[:300:3], color='tomato', s=15, zorder=5,
           label='Codon start (3-nt periodicity)', alpha=0.6)

ax.set_xlabel('Position (nt)')
ax.set_ylabel('Ribosome footprint count')
ax.set_title('Simulated Ribo-seq coverage (first 300 nt of GFP):\n3-nt periodicity = ribosome in-frame')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `find_splice_sites(pre_mrna)` from scratch. For a gene with 4 exons,
   how many GT-AG pairs are required at a minimum? Why are some GT-AG pairs false
   positives (not actual splice sites)?
2. RNA-seq reads that span an exon-exon junction have no contiguous match in the
   genome. Why must aligners like STAR use a 'split-read' strategy? What information
   is required to detect these reads?
3. Implement `codon_usage(cds)`. Compare the codon usage of GFP (codon-optimised)
   vs. a randomly shuffled version. Which has higher CAI?
4. Ribosome profiling shows a sharp peak in coverage at a specific codon. What
   biological interpretation would you consider? Name two experimental reasons that
   could also produce this artefact.

---
## Quiz — Active Recall

1. What is the GT-AG rule in splicing?
2. Name the three ribosome sites (A, P, E) and what each does.
3. What is codon usage bias, and why does it matter for recombinant protein expression?
4. Why does Pol II CTD phosphorylation state matter?
5. What is intron retention and how would it appear in RNA-seq data?

---
## Reflection

**Date completed:** ____________________

> *[You are aligning RNA-seq reads from a human sample and notice many reads with
> the pattern: 50 nt genome match / gap of ~1000 nt / 50 nt genome match. What is
> this read showing, and what alignment tool feature is detecting it?]*

---
*Next: `06_post_translational_modification_survey.ipynb`*